In [1]:
import pandas as pd

df = pd.read_csv("archive/fake.csv")
df.head()

FileNotFoundError: [Errno 2] No such file or directory: 'archive/fake.csv'

In [ ]:
df2 = pd.read_csv("archive/true.csv")
df["Type"] = "Fake"
df2["Type"] = "True"
df2.head()

,title,text,subject,date,Type
0,"As U.S. budget fight looms, Republicans flip t...",WASHINGTON (Reuters) - The head of a conservat...,politicsNews,"December 31, 2017",True
1,U.S. military to accept transgender recruits o...,WASHINGTON (Reuters) - Transgender people will...,politicsNews,"December 29, 2017",True
2,Senior U.S. Republican senator: 'Let Mr. Muell...,WASHINGTON (Reuters) - The special counsel inv...,politicsNews,"December 31, 2017",True
3,FBI Russia probe helped by Australian diplomat...,WASHINGTON (Reuters) - Trump campaign adviser ...,politicsNews,"December 30, 2017",True
4,Trump wants Postal Service to charge 'much mor...,SEATTLE/WASHINGTON (Reuters) - President Donal...,politicsNews,"December 29, 2017",True


In [ ]:
df = pd.concat([df, df2]).reset_index(drop=True)

In [ ]:
df.head()

,title,text,subject,date,Type
0,Donald Trump Sends Out Embarrassing New Year’...,Donald Trump just couldn t wish all Americans ...,News,"December 31, 2017",Fake
1,Drunk Bragging Trump Staffer Started Russian ...,House Intelligence Committee Chairman Devin Nu...,News,"December 31, 2017",Fake
2,Sheriff David Clarke Becomes An Internet Joke...,"On Friday, it was revealed that former Milwauk...",News,"December 30, 2017",Fake
3,Trump Is So Obsessed He Even Has Obama’s Name...,"On Christmas day, Donald Trump announced that ...",News,"December 29, 2017",Fake
4,Pope Francis Just Called Out Donald Trump Dur...,Pope Francis used his annual Christmas Day mes...,News,"December 25, 2017",Fake


In [ ]:
df["Type"].value_counts()

Type
Fake    23481
True    21417
Name: count, dtype: int64

## Preprocessing

In [ ]:
df["text"] = df["text"].astype(str)


In [ ]:
df["text"].dtype

dtype('O')

In [ ]:
import string
import nltk
from nltk.corpus import stopwords
nltk.download('stopwords')
from string import punctuation

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\meraj\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [ ]:
def remove_punctuation(text):
    text_nopunct = "".join([char for char in text if char not in string.punctuation])
    return text_nopunct

stopword = stopwords.words('english')
def remove_stopwords(text):
    text_nostop = " ".join([word for word in text.split() if word not in stopword])
    return text_nostop


In [ ]:
def string_lower(text):
    text_lower = text.lower()
    return text_lower

In [ ]:
!python3 -m spacy download en_core_web_sm --break-system-packages

Defaulting to user installation because normal site-packages is not writeable
Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 48.9 MB/s eta 0:00:00a 0:00:01
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


In [ ]:
import spacy
nlp = spacy.load('en_core_web_sm')
def lemmatization(text):
    doc = nlp(text)
    text_lem = " ".join([word.lemma_ for word in doc])
    return text_lem

In [ ]:
df["text"] = df["text"].apply(remove_punctuation)

In [ ]:
df["text"] = df["text"].apply(remove_stopwords)

In [ ]:
df["text"] = df["text"].apply(string_lower)

In [ ]:
df["text"] = df["text"].apply(lemmatization)

df.to_csv("cleaned_data.csv", index=False)

In [ ]:
df["word_count"] = df["text"].apply(lambda x: len(x.split()))
word_count = df["word_count"].sum()

In [ ]:
df

,title,text,subject,date,Type,word_count
0,Donald Trump Sends Out Embarrassing New Year’...,donald trump wish americans happy new year lea...,News,"December 31, 2017",0,299
1,Drunk Bragging Trump Staffer Started Russian ...,house intelligence committee chairman devin nu...,News,"December 31, 2017",0,185
2,Sheriff David Clarke Becomes An Internet Joke...,on friday reveal former milwaukee sheriff davi...,News,"December 30, 2017",0,359
3,Trump Is So Obsessed He Even Has Obama’s Name...,on christmas day donald trump announce would b...,News,"December 29, 2017",0,277
4,Pope Francis Just Called Out Donald Trump Dur...,pope francis use annual christmas day message ...,News,"December 25, 2017",0,223
...,...,...,...,...,...,...
44893,'Fully committed' NATO backs new U.S. approach...,brussels reuter nato ally tuesday welcome pres...,worldnews,"August 22, 2017",1,276
44894,LexisNexis withdrew two products from Chinese ...,london reuters lexisnexis provider legal regul...,worldnews,"August 22, 2017",1,79
44895,Minsk cultural hub becomes haven from authorities,minsk reuters in shadow disuse sovietera facto...,worldnews,"August 22, 2017",1,187
44896,Vatican upbeat on possibility of Pope Francis ...,moscow reuter vatican secretary state cardinal...,worldnews,"August 22, 2017",1,123


In [ ]:
df.head()

,title,text,subject,date,Type
0,Donald Trump Sends Out Embarrassing New Year’...,donald trump wish americans happy new year lea...,News,"December 31, 2017",Fake
1,Drunk Bragging Trump Staffer Started Russian ...,house intelligence committee chairman devin nu...,News,"December 31, 2017",Fake
2,Sheriff David Clarke Becomes An Internet Joke...,on friday reveal former milwaukee sheriff davi...,News,"December 30, 2017",Fake
3,Trump Is So Obsessed He Even Has Obama’s Name...,on christmas day donald trump announce would b...,News,"December 29, 2017",Fake
4,Pope Francis Just Called Out Donald Trump Dur...,pope francis use annual christmas day message ...,News,"December 25, 2017",Fake


In [1]:
import pandas as pd
df = pd.read_csv("cleaned_data.csv")

In [2]:
df["text"] = df["text"].fillna("").astype(str)
df["word_count"] = df["text"].apply(lambda x: len(str(x).split()))
word_count = df["word_count"].sum()

In [5]:
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences


tokenizer = Tokenizer(num_words=word_count, oov_token="<OOV>")
tokenizer.fit_on_texts(df["text"])
word_index = tokenizer.word_index
vocab_size = len(word_index) + 1
print("Vocabulary Size:", vocab_size)
sequences = tokenizer.texts_to_sequences(df["text"])
padded_sequences = pad_sequences(sequences, maxlen=120, padding='post', truncating='post')

le = LabelEncoder()
df["Type"] = le.fit_transform(df["Type"])
labels = df["Type"].values
X_train, X_test, y_train, y_test = train_test_split(padded_sequences, labels, test_size=0.2, random_state=42,shuffle=True)

Vocabulary Size: 216407


In [6]:
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Bidirectional, LSTM, Dense, Lambda, Add,
    Dropout, Attention, Concatenate, Flatten, SeparableConv1D,
    GlobalAveragePooling1D, LeakyReLU, Conv1D, MaxPooling1D,
    MultiHeadAttention,Embedding,BatchNormalization
)
from tensorflow.keras.callbacks import ReduceLROnPlateau
from tensorflow.keras.regularizers import l2

# Model parameters
input_shape = 120   # sequence_length x embedding_dim
num_classes = 2            # sentiment classes (e.g. positive, neutral, negative)

vocab_size = 216407        # Vocabulary size (adjust as needed)
embedding_dim = 128       # Word embedding size

# Input layer for tokenized text
input_tensor = Input(shape=(input_shape,))

# Embedding layer
x = Embedding(input_dim=vocab_size, output_dim=embedding_dim, input_length=input_shape)(input_tensor)

# First Bi-LSTM layer with attention
x = SeparableConv1D(32, kernel_size=3, activation="relu", padding="valid",
                    depth_multiplier=3, depthwise_initializer="glorot_uniform",
                    pointwise_initializer="glorot_uniform",
                    depthwise_regularizer=l2(1e-4), pointwise_regularizer=l2(1e-4))(x)
x = BatchNormalization()(x)
x = Dropout(0.2)(x)
x = MaxPooling1D(pool_size=2)(x)
attention_1 = Attention()([x, x])

mha_1 = MultiHeadAttention(num_heads=4, key_dim=16)(attention_1, attention_1)
x = Concatenate()([x, mha_1])
bi_lstm_1 = Bidirectional(LSTM(64, return_sequences=True, dropout=0.2, recurrent_dropout=0.2,
                               kernel_regularizer=l2(1e-4)))(x)

# Second Bi-LSTM layer with attention
# Second Conv1D layer after attention
x = SeparableConv1D(64, kernel_size=3, dilation_rate=2, activation="relu", padding="valid",
                    depth_multiplier=4, depthwise_initializer="glorot_uniform",
                    pointwise_initializer="glorot_uniform",
                    depthwise_regularizer=l2(1e-4), pointwise_regularizer=l2(1e-4))(bi_lstm_1)
x = BatchNormalization()(x)
x = Dropout(0.2)(x)
x = MaxPooling1D(pool_size=2)(x)
attention_2 = Attention()([x, x])

# Incremental num_heads: 6
mha_2 = MultiHeadAttention(num_heads=6, key_dim=32)(attention_2, attention_2)
x = Concatenate()([x, mha_2])

bi_lstm_2 = Bidirectional(LSTM(64, return_sequences=True, dropout=0.2, recurrent_dropout=0.2,
                               kernel_regularizer=l2(1e-4)))(x)

# Third Conv1D layer
x = SeparableConv1D(128, kernel_size=3,dilation_rate=3, activation="relu", padding="valid",
                    depth_multiplier=5, depthwise_initializer="glorot_uniform",
                    pointwise_initializer="glorot_uniform",
                    depthwise_regularizer=l2(1e-4), pointwise_regularizer=l2(1e-4))(bi_lstm_2)
x = BatchNormalization()(x)
x = Dropout(0.2)(x)
x = MaxPooling1D(pool_size=2)(x)

# Bi-LSTM after conv
bi_lstm_3 = Bidirectional(LSTM(64, return_sequences=True, dropout=0.2, recurrent_dropout=0.2,
                               kernel_regularizer=l2(1e-4)))(x)

# Fourth Conv1D layer
x = SeparableConv1D(256, kernel_size=3,dilation_rate=4, activation="relu", padding="valid",
                    depth_multiplier=6, depthwise_initializer="glorot_uniform",
                    pointwise_initializer="glorot_uniform",
                    depthwise_regularizer=l2(1e-4), pointwise_regularizer=l2(1e-4))(bi_lstm_3)
x = BatchNormalization()(x)
x = Dropout(0.2)(x)
x = MaxPooling1D(pool_size=2)(x)

# Bi-LSTM again
bi_lstm_4 = Bidirectional(LSTM(64, return_sequences=True, dropout=0.2, recurrent_dropout=0.2,
                               kernel_regularizer=l2(1e-4)))(x)

# Flatten
x = Flatten()(bi_lstm_4)

# Fully connected layers
x = Dense(128, activation="linear", kernel_regularizer=l2(1e-2))(x)
x = LeakyReLU()(x)
x = Dropout(0.2)(x)

x = Dense(64, activation="linear", kernel_regularizer=l2(1e-2))(x)
x = LeakyReLU()(x)
x = Dropout(0.2)(x)

# Output layer
output_tensor = Dense(num_classes, activation="softmax")(x)

# Create model
model = Model(inputs=input_tensor, outputs=output_tensor)

# Compile model
model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
lr_scheduler = ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-6)

# Summary
model.summary()


c:\Users\meraj\OneDrive\Desktop\New folder (3)\Fake_News_Project\.venv\Lib\site-packages\keras\src\layers\core\embedding.py:123: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 120)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, 120, 128)  │ 27,700,096 │ input_layer[0][0] │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ separable_conv1d    │ (None, 118, 32)   │     13,472 │ embedding[0][0]   │
│ (SeparableConv1D)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 118, 32)   │        128 │ separable_conv1d… │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 118, 32)   │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling1d       │ (None, 59, 32)    │          0 │ dropout[0][0]     │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ attention           │ (None, 59, 32)    │          0 │ max_pooling1d[0]… │
│ (Attention)         │                   │            │ max_pooling1d[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 59, 32)    │      8,416 │ attention[0][0],  │
│ (MultiHeadAttentio… │                   │            │ attention[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 59, 64)    │          0 │ max_pooling1d[0]… │
│ (Concatenate)       │                   │            │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional       │ (None, 59, 128)   │     66,048 │ concatenate[0][0] │
│ (Bidirectional)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ separable_conv1d_1  │ (None, 55, 64)    │     34,368 │ bidirectional[0]… │
│ (SeparableConv1D)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 55, 64)    │        256 │ separable_conv1d… │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_2 (Dropout) │ (None, 55, 64)    │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling1d_1     │ (None, 27, 64)    │          0 │ dropout_2[0][0]   │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ attention_1         │ (None, 27, 64)    │          0 │ max_pooling1d_1[… │
│ (Attention)         │                   │            │ max_pooling1d_1[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 27, 64)    │     49,792 │ attention_1[0][0… │
│ (MultiHeadAttentio… │                   │            │ attention_1[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_1       │ (None, 27, 128)   │          0 │ max_pooling1d_1[… │
│ (Concatenate)       │                   │            │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼─────────────────

 Total params: 28,544,130 (108.89 MB)

 Trainable params: 28,543,170 (108.88 MB)

 Non-trainable params: 960 (3.75 KB)

In [7]:
from tensorflow.keras.optimizers import Adam
model.compile(loss='sparse_categorical_crossentropy', 
              optimizer=Adam(learning_rate=0.001),  
              metrics=['accuracy'])

model.fit(X_train, y_train, epochs=2, batch_size=32,validation_split=0.2)

Epoch 1/2
111/898 ━━━━━━━━━━━━━━━━━━━━ 7:59 609ms/step - accuracy: 0.8730 - loss: 1.4574

KeyboardInterrupt: 

In [8]:
model.save("Sentiment_model_.keras")

In [9]:
predictions = model.predict(X_test)
predicted_labels = [tf.argmax(pred).numpy() for pred in predictions]
print(f"Predictions: {predicted_labels}")

from sklearn.metrics import classification_report,confusion_matrix
print(classification_report(y_test,predicted_labels))

import seaborn as sns
import matplotlib.pyplot as plt
sns.heatmap(confusion_matrix(y_test,predicted_labels))
plt.xlabel('Predicted Labels')
plt.ylabel('True Labels')
plt.title('Confusion Matrix')


281/281 ━━━━━━━━━━━━━━━━━━━━ 24s 73ms/step
Predictions: [np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int6

c:\Users\meraj\OneDrive\Desktop\New folder (3)\Fake_News_Project\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\meraj\OneDrive\Desktop\New folder (3)\Fake_News_Project\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\meraj\OneDrive\Desktop\New folder (3)\Fake_News_Project\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` par

ModuleNotFoundError: No module named 'seaborn'

In [ ]:
from tensorflow.keras.models import load_model
# model = load_model("Sentiment_model_.keras", compile=False)

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

test_text = ["NASA announced evidence of vast underground liquid water reservoirs in Mars' crust in August 2024, found using seismic data from the InSight lander. This discovery follows earlier 2015 confirmation of flowing liquid water on the surface via hydrated salts detected by the Mars Reconnaissance Orbiter and the July 2008 confirmation of subsurface water ice by the Phoenix lander"]
word_count = len(test_text[0].split())
tokenizer = Tokenizer(num_words=word_count)


In [ ]:
word_count

58

In [3]:
import tensorflow as tf

In [ ]:

sequences = tokenizer.texts_to_sequences(test_text)
padded_sequences = pad_sequences(sequences, maxlen=120, padding='post', truncating='post')

predictions = model.predict(padded_sequences)
predicted_labels = [tf.argmax(pred).numpy() for pred in predictions]
print(f"Predictions: {predicted_labels}")


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 168ms/step
Predictions: [0]


In [ ]:
type(test_text)

list

In [ ]:
# convert_model.py

import tensorflow as tf
import json

# ... (Your model loading and tokenizer saving code remains the same) ...

# --- 3. Convert the Keras Model to TFLite ---
# The conversion process uses the TFLiteConverter
converter = tf.lite.TFLiteConverter.from_keras_model(model)

# ==============================================================================
# ▼▼▼ ADD THESE TWO LINES TO FIX THE ERROR ▼▼▼
# ==============================================================================

# This enables the converter to use TensorFlow ops when a TFLite op is not available.
converter.target_spec.supported_ops = [
  tf.lite.OpsSet.TFLITE_BUILTINS, # Enable TFLite builtin ops.
  tf.lite.OpsSet.SELECT_TF_OPS    # Enable TensorFlow ops.
]

# This disables the experimental feature that tries to lower tensor list ops.
# It's recommended to disable it when using Flex delegation for LSTMs.
converter._experimental_lower_tensor_list_ops = False

# ==============================================================================
# ▲▲▲ END OF FIX ▲▲▲
# ==============================================================================


# Apply default optimizations to reduce model size
converter.optimizations = [tf.lite.Optimize.DEFAULT]

# Convert the model
tflite_model = converter.convert()

# --- 4. Save the TFLite Model ---
# Write the converted model to a .tflite file
with open('model.tflite', 'wb') as f:
    f.write(tflite_model)

print("\nModel conversion complete with Flex Ops enabled!")
print("The following files have been created:")
print("- model.tflite")
print("- tokenizer.json")
print("- metadata.json")

INFO:tensorflow:Assets written to: /tmp/tmp0wcwdxr_/assets


INFO:tensorflow:Assets written to: /tmp/tmp0wcwdxr_/assets


Saved artifact at '/tmp/tmp0wcwdxr_'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 120), dtype=tf.float32, name='keras_tensor')
Output Type:
  TensorSpec(shape=(None, 2), dtype=tf.float32, name=None)
Captures:
  138967560500624: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138967560501200: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138967560502160: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138967560502736: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138967560503120: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138967560502928: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138967560501392: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138967560500816: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138967560501776: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138967640998928: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138967640999504: Tenso

W0000 00:00:1759294790.732584    7018 tf_tfl_flatbuffer_helpers.cc:365] Ignored output_format.
W0000 00:00:1759294790.732607    7018 tf_tfl_flatbuffer_helpers.cc:368] Ignored drop_control_dependency.
2025-10-01 10:29:50.733040: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /tmp/tmp0wcwdxr_
2025-10-01 10:29:50.740759: I tensorflow/cc/saved_model/reader.cc:52] Reading meta graph with tags { serve }
2025-10-01 10:29:50.740774: I tensorflow/cc/saved_model/reader.cc:147] Reading SavedModel debug info (if present) from: /tmp/tmp0wcwdxr_
I0000 00:00:1759294790.851615    7018 mlir_graph_optimization_pass.cc:425] MLIR V1 optimization pass is not enabled
2025-10-01 10:29:50.861768: I tensorflow/cc/saved_model/loader.cc:236] Restoring SavedModel bundle.
2025-10-01 10:29:51.322024: I tensorflow/cc/saved_model/loader.cc:220] Running initialization op on SavedModel bundle at path: /tmp/tmp0wcwdxr_
2025-10-01 10:29:51.509284: I tensorflow/cc/saved_model/loader.cc:471] SavedModel 


Model conversion complete with Flex Ops enabled!
The following files have been created:
- model.tflite
- tokenizer.json
- metadata.json


In [5]:
import pandas as pd
from tensorflow.keras.preprocessing.text import Tokenizer
import pickle

# ডেটা লোড করা
df = pd.read_csv("cleaned_data.csv")
df["text"] = df["text"].fillna("").astype(str)

# টোকেনাইজার তৈরি করা
word_count = df["text"].apply(lambda x: len(str(x).split())).sum()
tokenizer = Tokenizer(num_words=word_count, oov_token="<OOV>")
tokenizer.fit_on_texts(df["text"])

# টোকেনাইজার সেভ করা
with open("tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)

print("✅ Tokenizer saved successfully!")

✅ Tokenizer saved successfully!


In [7]:
import pandas as pd
import numpy as np
import tensorflow as tf
import pickle
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Bidirectional, LSTM, Dense, Dropout, Attention, Concatenate, Flatten, SeparableConv1D, LeakyReLU, MaxPooling1D, MultiHeadAttention, Embedding, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l2

print("1. Cleaning data...")
df = pd.read_csv("cleaned_data.csv")
df["text"] = df["text"].fillna("").astype(str)
# Removing the bias ('reuters') from the dataset
df['text'] = df['text'].str.replace('reuters', ' ', case=False)
df['text'] = df['text'].str.replace('reuter', ' ', case=False)

print("2. Creating and saving tokenizer...")
word_count = df["text"].apply(lambda x: len(str(x).split())).sum()
tokenizer = Tokenizer(num_words=word_count, oov_token="<OOV>")
tokenizer.fit_on_texts(df["text"])
vocab_size = len(tokenizer.word_index) + 1
sequences = tokenizer.texts_to_sequences(df["text"])
padded_sequences = pad_sequences(sequences, maxlen=120, padding='post', truncating='post')

with open("tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)

print("3. Preparing data for training...")
le = LabelEncoder()
labels = le.fit_transform(df["Type"])
X_train, X_test, y_train, y_test = train_test_split(padded_sequences, labels, test_size=0.2, random_state=42, shuffle=True)

print("4. Building the model...")
input_shape = 120
embedding_dim = 128
num_classes = 2

input_tensor = Input(shape=(input_shape,))
x = Embedding(input_dim=vocab_size, output_dim=embedding_dim)(input_tensor)

# Layer 1
x = SeparableConv1D(32, kernel_size=3, activation="relu", padding="valid", depth_multiplier=3, depthwise_regularizer=l2(1e-4), pointwise_regularizer=l2(1e-4))(x)
x = BatchNormalization()(x)
x = Dropout(0.2)(x)
x = MaxPooling1D(pool_size=2)(x)
attention_1 = Attention()([x, x])
mha_1 = MultiHeadAttention(num_heads=4, key_dim=16)(attention_1, attention_1)
x = Concatenate()([x, mha_1])
bi_lstm_1 = Bidirectional(LSTM(64, return_sequences=True, dropout=0.2, recurrent_dropout=0.2, kernel_regularizer=l2(1e-4)))(x)

# Layer 2
x = SeparableConv1D(64, kernel_size=3, dilation_rate=2, activation="relu", padding="valid", depth_multiplier=4, depthwise_regularizer=l2(1e-4), pointwise_regularizer=l2(1e-4))(bi_lstm_1)
x = BatchNormalization()(x)
x = Dropout(0.2)(x)
x = MaxPooling1D(pool_size=2)(x)
attention_2 = Attention()([x, x])
mha_2 = MultiHeadAttention(num_heads=6, key_dim=32)(attention_2, attention_2)
x = Concatenate()([x, mha_2])
bi_lstm_2 = Bidirectional(LSTM(64, return_sequences=True, dropout=0.2, recurrent_dropout=0.2, kernel_regularizer=l2(1e-4)))(x)

# Layer 3
x = SeparableConv1D(128, kernel_size=3, dilation_rate=3, activation="relu", padding="valid", depth_multiplier=5, depthwise_regularizer=l2(1e-4), pointwise_regularizer=l2(1e-4))(bi_lstm_2)
x = BatchNormalization()(x)
x = Dropout(0.2)(x)
x = MaxPooling1D(pool_size=2)(x)
bi_lstm_3 = Bidirectional(LSTM(64, return_sequences=True, dropout=0.2, recurrent_dropout=0.2, kernel_regularizer=l2(1e-4)))(x)

# Layer 4
x = SeparableConv1D(256, kernel_size=3, dilation_rate=4, activation="relu", padding="valid", depth_multiplier=6, depthwise_regularizer=l2(1e-4), pointwise_regularizer=l2(1e-4))(bi_lstm_3)
x = BatchNormalization()(x)
x = Dropout(0.2)(x)
x = MaxPooling1D(pool_size=2)(x)
bi_lstm_4 = Bidirectional(LSTM(64, return_sequences=True, dropout=0.2, recurrent_dropout=0.2, kernel_regularizer=l2(1e-4)))(x)

# Flatten & Dense
x = Flatten()(bi_lstm_4)
x = Dense(128, activation="linear", kernel_regularizer=l2(1e-2))(x)
x = LeakyReLU()(x)
x = Dropout(0.2)(x)
x = Dense(64, activation="linear", kernel_regularizer=l2(1e-2))(x)
x = LeakyReLU()(x)
x = Dropout(0.2)(x)

output_tensor = Dense(num_classes, activation="softmax")(x)

model = Model(inputs=input_tensor, outputs=output_tensor)
model.compile(loss='sparse_categorical_crossentropy', optimizer=Adam(learning_rate=0.001), metrics=['accuracy'])

print("5. Training the model (this may take some time)...")
model.fit(X_train, y_train, epochs=2, batch_size=32, validation_split=0.2)

print("6. Saving the new model...")
model.save("Sentiment_model_.keras")
print("✅ All tasks completed successfully!")

1. Cleaning data...
2. Creating and saving tokenizer...
3. Preparing data for training...
4. Building the model...
5. Training the model (this may take some time)...
Epoch 1/2
898/898 ━━━━━━━━━━━━━━━━━━━━ 499s 531ms/step - accuracy: 0.9536 - loss: 0.4156 - val_accuracy: 0.9857 - val_loss: 0.2624
Epoch 2/2
898/898 ━━━━━━━━━━━━━━━━━━━━ 484s 539ms/step - accuracy: 0.9888 - loss: 0.1114 - val_accuracy: 0.9882 - val_loss: 0.0958
6. Saving the new model...
✅ All tasks completed successfully!
